# AudioCraft - Google Colab

Quick setup for music generation with Python 3.12

In [ ]:
# Installation
import os
if os.path.exists('/content/audiocraft'):
    !rm -rf /content/audiocraft
!git clone -b colab-python312-support https://github.com/loudmantrade/audiocraft.git /content/audiocraft
!apt-get update -qq && apt-get install -y -qq libavformat-dev libavcodec-dev libavdevice-dev libavutil-dev libswscale-dev ffmpeg

# Copy Python 3.12 compatible files
!cp /content/audiocraft/setup_py312.py /content/audiocraft/setup.py

# Verify Python version constraint
!grep "REQUIRES_PYTHON" /content/audiocraft/setup.py

# Install PyTorch (latest version for Python 3.12)
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118

# Install xformers for transformer acceleration
!pip install -q xformers

# Install core dependencies first
!pip install -q av einops transformers huggingface_hub sentencepiece num2words hydra-core omegaconf gradio julius encodec demucs torchmetrics librosa

# Install audiocraft (warnings about numpy are safe to ignore)
!cd /content/audiocraft && pip install -e .

# Verify installation
!python -c "import audiocraft; print('✓ AudioCraft installed:', audiocraft.__version__)"
!python -c "from audiocraft.models import MusicGen; print('✓ MusicGen module available')"

Cloning into '/content/audiocraft'...
remote: Enumerating objects: 2045, done.
remote: Counting objects: 100% (684/684), done.
remote: Compressing objects: 100% (136/136), done.
remote: Total 2045 (delta 569), reused 570 (delta 548), pack-reused 1361 (from 1)
Receiving objects: 100% (2045/2045), 24.75 MiB | 25.58 MiB/s, done.
Resolving deltas: 100% (1218/1218), done.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
REQUIRES_PYTHON = '>=3.8.0,<3.13'
    python_requires=REQUIRES_PYTHON,
  Preparing metadata (setup.py) ... done


In [ ]:
# Load model
# Note: NumPy version warnings are safe to ignore - AudioCraft uses numpy 1.26.4
from audiocraft.models import MusicGen
import torchaudio

model = MusicGen.get_pretrained('facebook/musicgen-small')
print('Model loaded successfully!')

ModuleNotFoundError: No module named 'audiocraft.models'

In [ ]:
# Generate music
model.set_generation_params(duration=10)

descriptions = [
    'upbeat electronic dance music with synthesizers',
    'calm acoustic guitar melody'
]

wav = model.generate(descriptions)

# Save files
for idx, one_wav in enumerate(wav):
    audio_path = f'generated_{idx}.wav'
    torchaudio.save(audio_path, one_wav.cpu(), model.sample_rate)
    print(f'Saved: {audio_path}')

In [ ]:
# Play audio
from IPython.display import Audio

for idx, desc in enumerate(descriptions):
    print(f'Track {idx}: {desc}')
    display(Audio(f'generated_{idx}.wav'))